In [1]:
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from helper_functions import protocol_noise_free as nf
from helper_functions import protocol_phase_diffusion as pd
from scipy.optimize import curve_fit

In [2]:
from importlib import reload
def reload_protocols():
    reload(pd)

## cs

## Produce data

In [3]:
sigmas = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]

alpha_grid = np.linspace(0, np.sqrt(2), 40)
num_samples = 3000

for i, sigma in enumerate(sigmas):

    p_err_cs = pd.phase_diffused_cs(alpha_grid, sigma, num_samples)

    print(f"\rProgress: {i+1}/{len(sigmas)}", end="", flush=True)

    np.savez(f"data/phase_diff/perr_data_phase_diff_cs_a{len(alpha_grid)}_S{num_samples}_sigma{sigma}.npz", alpha_grid = alpha_grid, p_err_cs = p_err_cs, sigma = sigma)

Progress: 10/10

## Load data

In [ ]:
data = np.load(f"data/perr_data_phase_diff_cs_a30_S3000_sigma0.1.npz")

alpha_cs = data["alpha_grid"]
p_err_cs = data["p_err_cs"]
sigma_cs = data["sigma"]

In [ ]:
fig = plt.figure(figsize=(10,4), dpi=150)
N = alpha_cs**2
for i, p_err in enumerate(p_err_cs_list):
    plt.scatter(N, p_err, label = fr'$\sigma$={sigmas_cs[i]:.2f}', s=4.5)
plt.xlabel('N')
plt.ylabel('P_err')
plt.plot(N, nf.perr_homodyne(N, 0), label=r'P_err theoretical for $\sigma$=0')
plt.yscale('log')
plt.legend()

## DSS

## Produce data

In [ ]:
# Parameters
N = np.linspace(0, 2.0, 40)
beta = np.linspace(0.0, 1.0, 40)
'''
sigma = 0.2
'''
num_samples = 10000

for sigma in [0.5, 0.6, 0.7]:

    print(f"σ = {sigma}")
    
    # Compute error probabilities
    p_err_dss = pd.phase_diffused_dss(N, beta, sigma, num_samples)
    # Create meshgrid for plotting
    N_grid, beta_grid = np.meshgrid(N, beta, indexing="ij")
    #save data
    np.savez(f"data/perr_data_phase_diff_dss_N{len(N)}_b{len(beta)}_S{num_samples}_sigma{sigma}.npz", N=N, beta = beta, p_err_dss=p_err_dss, sigma = sigma)


## Load data

In [ ]:
data = np.load(f"data/perr_data_phase_diff_dss_N40_b40_S10000_sigma0.1.npz")

N_dss = data["N"]
beta_dss = data["beta"]
p_err_dss = data["p_err_dss"]
sigma_dss = data["sigma"]

## Visualize Data

In [ ]:
# Flatten grid for scatter points
colors = ["blue", "red", "green", "orange", "purple", "brown", "pink", "gray", "olive", "cyan"]

fig = go.Figure()

sigmas = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]

for i, sigma in enumerate(sigmas):

    data = np.load(f"data/perr_data_phase_diff_dss_N40_b40_S10000_sigma{sigma}.npz")
    N_dss = data["N"]
    beta_dss = data["beta"]
    p_err_dss = data["p_err_dss"]
    sigma_dss = data["sigma"]

    N_grid, beta_grid = np.meshgrid(N_dss, beta_dss, indexing="ij")

    fig.add_trace(
        go.Scatter3d(
            x=N_grid.ravel(),
            y=beta_grid.ravel(),
            z=p_err_dss.ravel(),
            mode="markers",
            marker=dict(
                size=3,
                color=colors[i]
            ),
            name=f"σ={sigma}"
        )
    )

fig.update_layout(
    title=rf"Phase-diffused DSS (σ={sigma_dss})", 
    scene=dict(xaxis_title="N", yaxis_title=r"β", 
               zaxis=dict(title="P_err", type='log'),
               aspectmode="cube"), 
               width=900, height=800)

fig.show()

## Fit

In [ ]:
reload_protocols()
sigmas = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
pd.fit_homodyne_perr(sigmas)